In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

PROVINCE_COLORS = {
    'AJK':              '#4e9a8f',
    'Balochistan':      '#e07b39',
    'Gilgit-Baltistan': '#9b59b6',
    'ICT':              '#7f8c8d',
    'KPK':              '#2ca02c',
    'Punjab':           '#1f77b4',
    'Sindh':            '#d62728',
}

raw = pd.read_csv('../Data:/district_summary.csv')
df  = raw[raw['total_children_assessed'] >= 100].copy()

# DATA QUALITY: mean_earning_members has 5 districts with implausible values
# (e.g. Lower Dir = 1434.03), almost certainly data entry errors.
# Clip at 10 for display; this drops 5 points from that scatter only.
EARNING_CAP = 10
df['mean_earning_members_capped'] = df['mean_earning_members'].clip(upper=EARNING_CAP)
n_earning_outliers = (df['mean_earning_members'] > EARNING_CAP).sum()

print(f'Districts after filter: {len(df)}')
print(f'mean_earning_members > {EARNING_CAP} (capped for display): {n_earning_outliers}')
print(f'pct_receives_bisp missing: {df["pct_receives_bisp"].isna().sum()} / {len(df)}')

## Figure 1 — Socioeconomic Correlates of the Govt–Private Arithmetic Gap

The 2×2 grid below tests whether district-level socioeconomic characteristics predict the size of the arithmetic gap between private and government school students. Each panel shows a scatter plot with a least-squares regression line; Pearson *r* and two-tailed *p*-value are annotated in the corner.

**Data notes before reading:**
- **`pct_receives_bisp`**: only 50 of 149 districts have a non-missing value for BISP receipt. The correlation and regression line are estimated on this partial sub-sample, which skews toward provinces where BISP data was collected consistently (primarily Punjab and Sindh). Treat with caution.
- **`mean_earning_members`**: five districts have implausible values (the maximum in the raw data is 1,434 — almost certainly a data-entry error). The scatter clips values above 10 to preserve readability; these five points are excluded from the panel.

In [ ]:
SES_VARS = [
    ('pct_has_internet',           'Households with internet (%)'),
    ('pct_has_smartphone',         'Households with smartphone (%)'),
    ('pct_receives_bisp',          'Households receiving BISP (%)'),
    ('mean_earning_members_capped', f'Mean earning members (capped at {EARNING_CAP})'),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

legend_handles = [
    mpatches.Patch(facecolor=col, label=prov)
    for prov, col in PROVINCE_COLORS.items()
]

for ax, (xcol, xlabel) in zip(axes.flat, SES_VARS):
    sub = df[['arithmetic_gap', xcol, 'province']].dropna()

    # Province-coloured scatter
    for prov, grp in sub.groupby('province'):
        ax.scatter(
            grp[xcol], grp['arithmetic_gap'],
            color=PROVINCE_COLORS.get(prov, '#888888'),
            s=28, alpha=0.78, linewidths=0,
        )

    # OLS regression line
    slope, intercept, r_val, p_val, _ = stats.linregress(
        sub[xcol].astype(float), sub['arithmetic_gap'].astype(float)
    )
    x0, x1 = sub[xcol].min(), sub[xcol].max()
    ax.plot([x0, x1], [slope * x0 + intercept, slope * x1 + intercept],
            color='#222222', linewidth=1.6, zorder=5)

    # Annotation box
    p_str = f'p < 0.001' if p_val < 0.001 else f'p = {p_val:.3f}'
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else ''))
    ax.text(
        0.97, 0.97,
        f'r = {r_val:.2f}{sig}\n{p_str}\nn = {len(sub)}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                  alpha=0.88, edgecolor='#cccccc', linewidth=0.8),
    )

    ax.axhline(0, color='#aaaaaa', linewidth=0.8, linestyle='--', zorder=0)
    ax.set_xlabel(xlabel, labelpad=6, fontsize=10)
    ax.set_ylabel('Arithmetic gap\n(private − government)', labelpad=6, fontsize=10)
    sns.despine(ax=ax)

# Shared legend below
fig.legend(
    handles=legend_handles, ncol=7,
    loc='lower center', bbox_to_anchor=(0.5, -0.04),
    frameon=False, fontsize=9,
)
fig.suptitle('Socioeconomic Correlates of the Govt–Private Arithmetic Gap',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

fig.savefig(FIGURES / '04a_ses_scatter_grid.png', bbox_inches='tight')
fig.savefig(FIGURES / '04a_ses_scatter_grid.pdf', bbox_inches='tight')
plt.show()

# Summary table of correlations
rows = []
for xcol, xlabel in SES_VARS:
    sub = df[['arithmetic_gap', xcol]].dropna()
    r_val, p_val = stats.pearsonr(sub[xcol].astype(float), sub['arithmetic_gap'].astype(float))
    rows.append({'Variable': xlabel, 'n': len(sub), 'r': round(r_val, 3), 'p': round(p_val, 4)})
print(pd.DataFrame(rows).to_string(index=False))

## Figure 2 — Full Numeric Correlation Heatmap

The heatmap below shows pairwise Pearson correlations for all 29 numeric columns in the district summary dataset, using the lower triangle only (upper triangle is redundant). Several patterns are worth noting before drawing inference:

- **Constructed relationships**: `arithmetic_gap`, `urdu_gap`, and `english_gap` are algebraic differences of the school-type means (e.g., `arithmetic_gap = pvt_mean_arithmetic − govt_mean_arithmetic`), so correlations involving these three are partially mechanical.
- **Proficiency % vs. mean score**: `govt_pct_division`, `pvt_pct_division`, and similar binary-proficiency rates are derived from the same assessment as the mean scores and will be strongly collinear.
- **Substantive correlations of interest**: relationships between the SES/infrastructure variables (`pct_has_electricity`, `pct_has_internet`, `pct_receives_bisp`) and the absolute government score (`govt_mean_arithmetic`) are the most policy-relevant entries in this matrix.

In [ ]:
numeric_df = df.select_dtypes('number').drop(columns=['mean_earning_members_capped'])

# Shorten column names to fit the heatmap
label_map = {
    'total_children_assessed':      'N assessed',
    'n_government':                 'n govt',
    'n_private':                    'n pvt',
    'pct_female':                   '% female',
    'mean_arithmetic_score':        'arith (all)',
    'mean_urdu_reading_score':      'urdu (all)',
    'mean_english_reading_score':   'eng (all)',
    'govt_mean_arithmetic':         'arith govt',
    'pvt_mean_arithmetic':          'arith pvt',
    'arithmetic_gap':               'arith gap',
    'govt_mean_urdu':               'urdu govt',
    'pvt_mean_urdu':                'urdu pvt',
    'urdu_gap':                     'urdu gap',
    'govt_mean_english':            'eng govt',
    'pvt_mean_english':             'eng pvt',
    'english_gap':                  'eng gap',
    'pct_can_do_division':          '% division',
    'govt_pct_division':            '% div govt',
    'pvt_pct_division':             '% div pvt',
    'pct_can_read_story_urdu':      '% story urd',
    'govt_pct_story_urdu':          '% story urd G',
    'pvt_pct_story_urdu':           '% story urd P',
    'pct_can_read_sentences_english':'% sent eng',
    'pct_has_electricity':          '% electric',
    'pct_has_internet':             '% internet',
    'pct_has_smartphone':           '% smartphone',
    'pct_receives_bisp':            '% BISP',
    'pct_flood_affected':           '% flood',
    'mean_earning_members':         'earn members',
}
corr = numeric_df.rename(columns=label_map).corr()

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # mask upper triangle

fig, ax = plt.subplots(figsize=(17, 14))
sns.heatmap(
    corr,
    mask=mask,
    cmap='RdBu_r',
    vmin=-1, vmax=1, center=0,
    annot=True, fmt='.1f',
    annot_kws={'size': 6},
    linewidths=0.3, linecolor='white',
    square=True,
    cbar_kws={'label': 'Pearson r', 'shrink': 0.6},
    ax=ax,
)
ax.set_title('Pairwise Correlations — District Summary (all numeric variables)',
             fontsize=13, fontweight='bold', pad=14)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)
plt.tight_layout()

fig.savefig(FIGURES / '04b_correlation_heatmap.png', bbox_inches='tight')
fig.savefig(FIGURES / '04b_correlation_heatmap.pdf', bbox_inches='tight')
plt.show()

## Figure 3 — Government School Arithmetic Score vs Household Electrification

Electrification rate (`pct_has_electricity`) is used here as a composite proxy for district-level infrastructure development and socioeconomic status — areas without reliable electricity also tend to have weaker road networks, lower teacher retention, and higher poverty rates. The scatter below plots the **absolute government school arithmetic score** (rather than the gap) against electrification, which removes the confounding influence of private school quality and focuses squarely on the state of public education.

The five lowest-scoring government districts are labelled directly on the chart. These are the districts where public-school learning outcomes are critically low and where infrastructure investment and teacher-deployment policy should be concentrated.

In [ ]:
elec_df = df[['district', 'province', 'govt_mean_arithmetic', 'pct_has_electricity']].dropna()

# 5 lowest-scoring government districts
worst5 = elec_df.nsmallest(5, 'govt_mean_arithmetic')

# OLS line across full filtered set
slope, intercept, r_val, p_val, _ = stats.linregress(
    elec_df['pct_has_electricity'], elec_df['govt_mean_arithmetic']
)

fig, ax = plt.subplots(figsize=(9, 6))

# Province-coloured scatter
for prov, grp in elec_df.groupby('province'):
    ax.scatter(
        grp['pct_has_electricity'], grp['govt_mean_arithmetic'],
        color=PROVINCE_COLORS.get(prov, '#888888'),
        s=32, alpha=0.8, linewidths=0, label=prov,
    )

# Regression line
x_range = np.linspace(elec_df['pct_has_electricity'].min(),
                       elec_df['pct_has_electricity'].max(), 100)
ax.plot(x_range, slope * x_range + intercept,
        color='#222222', linewidth=1.5, zorder=5)

# Highlight worst-5 with ring and label
ax.scatter(
    worst5['pct_has_electricity'], worst5['govt_mean_arithmetic'],
    s=90, facecolors='none',
    edgecolors=[PROVINCE_COLORS[p] for p in worst5['province']],
    linewidths=1.8, zorder=6,
)

# Manual label offsets to avoid overlap
label_offsets = {
    'Chagai':              (-2,  0.12),
    'Kohlu':               ( 1.5, 0.05),
    'Khairpur':            (-2,  0.12),
    'Matiari':             ( 1.5, 0.05),
    'Hub':                 (-12, 0.08),
}
for _, row in worst5.iterrows():
    dx, dy = label_offsets.get(row['district'], (1.5, 0.06))
    ax.annotate(
        row['district'],
        xy=(row['pct_has_electricity'], row['govt_mean_arithmetic']),
        xytext=(row['pct_has_electricity'] + dx, row['govt_mean_arithmetic'] + dy),
        fontsize=8.5, color='#222222',
        arrowprops=dict(arrowstyle='-', color='#888888', lw=0.8),
    )

# Stat annotation
p_str = 'p < 0.001' if p_val < 0.001 else f'p = {p_val:.3f}'
ax.text(0.03, 0.97,
        f'r = {r_val:.2f}\n{p_str}\nn = {len(elec_df)}',
        transform=ax.transAxes, ha='left', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                  alpha=0.88, edgecolor='#cccccc', linewidth=0.8))

ax.set_xlabel('Households with electricity (%)', labelpad=8)
ax.set_ylabel('Govt school mean arithmetic score', labelpad=8)
ax.set_title('Government School Learning vs District Electrification Rate',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=False, fontsize=9, loc='lower right',
          bbox_to_anchor=(1.0, 0.0))
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '04c_govt_score_vs_electricity.png', bbox_inches='tight')
fig.savefig(FIGURES / '04c_govt_score_vs_electricity.pdf', bbox_inches='tight')
plt.show()

print('\n5 lowest-scoring government districts:')
print(worst5[['district','province','govt_mean_arithmetic','pct_has_electricity']].to_string(index=False))